# Challenge AI Engineer Intern Test

Notebook ini disusun agar siap dijalankan di Google Colab dan dapat langsung digunakan sebagai submission untuk tes AI Engineer Intern.

## Ringkasan Tugas

- **Bagian 1 yang dipilih:** Soal 1B - Aplikasi Chatbot FAQ melalui CLI/Terminal
- **Bagian 2:** jawaban teori dasar AI secara ringkas, jelas, dan terstruktur

## Asumsi Implementasi

- Chatbot dijalankan dalam gaya CLI melalui `input()` di notebook, sehingga alurnya tetap seperti terminal.
- Model yang dipakai berasal dari **Ollama Cloud**.
- Bot hanya boleh menjawab berdasarkan konteks dari `faq.txt`.


## Bagian 1B - Chatbot FAQ dengan Ollama Cloud

Alur kerja notebook ini adalah sebagai berikut:

1. Menginstal library yang dibutuhkan.
2. Mengisi API key Ollama Cloud.
3. Menampilkan daftar model cloud dan meminta user memilih model.
4. Membuat file `faq.txt` sebagai konteks FAQ.
5. Memuat FAQ dan mencari jawaban yang paling relevan.
6. Mengirim konteks yang relevan ke model Ollama Cloud.
7. Menjalankan loop percakapan CLI/Terminal.


In [ ]:
# Install library yang dibutuhkan di Colab.
!pip -q install ollama requests


In [ ]:
import os
import re
from difflib import SequenceMatcher
from pathlib import Path
from getpass import getpass

import requests
from ollama import Client

print('Library siap digunakan.')


### 1) Siapkan Ollama Cloud API Key

Ollama Cloud membutuhkan autentikasi. Jika `OLLAMA_API_KEY` belum ada di environment, notebook akan meminta Anda memasukkannya secara aman.

Dokumentasi Ollama menjelaskan bahwa cloud model dijalankan lewat `ollama.com` dan akses API-nya membutuhkan key.


In [ ]:
if not os.getenv('OLLAMA_API_KEY'):
    os.environ['OLLAMA_API_KEY'] = getpass('Masukkan OLLAMA_API_KEY dari Ollama Cloud: ')

OLLAMA_API_KEY = os.environ['OLLAMA_API_KEY'].strip()
OLLAMA_HEADERS = {'Authorization': f'Bearer {OLLAMA_API_KEY}'}
print('API key terset.')


### 2) Pilih Model Ollama Cloud

Notebook ini mencoba mengambil daftar model cloud dari `https://ollama.com/api/tags`. Jika daftar tidak bisa diambil, notebook tetap menyediakan pilihan contoh model cloud.

Contoh model cloud yang umum dipakai:

- `gpt-oss:120b-cloud`
- `glm-5:cloud`
- `kimi-k2.5:cloud`


In [ ]:
DEFAULT_CLOUD_MODELS = [
    'gpt-oss:120b-cloud',
    'glm-5:cloud',
    'kimi-k2.5:cloud',
]

def fetch_cloud_models():
    try:
        response = requests.get(
            'https://ollama.com/api/tags',
            headers=OLLAMA_HEADERS,
            timeout=30,
        )
        response.raise_for_status()
        payload = response.json()
        models = [item.get('name') for item in payload.get('models', []) if item.get('name')]
        return models or DEFAULT_CLOUD_MODELS
    except Exception as exc:
        print(f'Gagal mengambil daftar model cloud: {exc}')
        return DEFAULT_CLOUD_MODELS

available_models = fetch_cloud_models()
print('\nDaftar model yang tersedia:')
for idx, model_name in enumerate(available_models, start=1):
    print(f'{idx}. {model_name}')

choice = input('\nPilih model dengan nomor, atau ketik nama model langsung: ').strip()
if choice.isdigit() and 1 <= int(choice) <= len(available_models):
    selected_model = available_models[int(choice) - 1]
else:
    selected_model = choice or available_models[0]

print(f'Model aktif: {selected_model}')


### 3) Buat File `faq.txt`

Setiap baris berisi satu pasangan pertanyaan dan jawaban. Format yang dipakai adalah:

`pertanyaan | jawaban`

Struktur ini mudah dibaca dan mudah diproses untuk mencari jawaban yang relevan.


In [ ]:
faq_entries = [
    ('Apa itu notebook ini?', 'Notebook ini adalah submission untuk tes AI Engineer Intern yang berisi chatbot FAQ berbasis Ollama Cloud.'),
    ('Bagaimana cara menjalankan chatbot?', 'Jalankan cell dari atas ke bawah, pilih model, lalu ketik pertanyaan di bagian CLI/Terminal notebook.'),
    ('Apa fungsi file faq.txt?', 'File faq.txt menyimpan konteks FAQ yang menjadi satu-satunya sumber jawaban chatbot.'),
    ('Apa yang dilakukan jika pertanyaan tidak ada di FAQ?', 'Chatbot harus menjawab: Maaf, saya tidak dapat membantu dengan pertanyaan itu.'),
    ('Apakah chatbot ini boleh menjawab dari luar konteks?', 'Tidak. Chatbot hanya boleh menjawab berdasarkan konteks yang ada di faq.txt.'),
    ('Model apa yang bisa dipakai?', 'Anda bisa memilih model cloud yang tersedia, misalnya gpt-oss:120b-cloud, glm-5:cloud, atau kimi-k2.5:cloud jika tersedia di akun Anda.'),
]

faq_path = Path('faq.txt')
faq_path.write_text('\n'.join(f'{q} | {a}' for q, a in faq_entries), encoding='utf-8')
print(f'faq.txt dibuat di: {faq_path.resolve()}')


### 4) Load FAQ dan Cari Jawaban yang Relevan

Agar bot tetap ketat, notebook ini melakukan pencarian FAQ paling mirip terlebih dahulu. Jika tingkat kemiripan terlalu rendah, bot langsung memberikan jawaban fallback tanpa menebak.


In [ ]:
def load_faq(path='faq.txt'):
    pairs = []
    for line in Path(path).read_text(encoding='utf-8').splitlines():
        line = line.strip()
        if not line or '|' not in line:
            continue
        question, answer = [part.strip() for part in line.split('|', 1)]
        pairs.append((question, answer))
    return pairs


def tokenize(text):
    return set(re.findall(r'[a-z0-9]+', text.lower()))


def similarity(a, b):
    tokens_a = tokenize(a)
    tokens_b = tokenize(b)
    if not tokens_a or not tokens_b:
        return 0.0
    jaccard = len(tokens_a & tokens_b) / len(tokens_a | tokens_b)
    ratio = SequenceMatcher(None, a.lower(), b.lower()).ratio()
    return (0.65 * jaccard) + (0.35 * ratio)


def find_best_faq_match(question, faq_pairs):
    scored = []
    for faq_question, faq_answer in faq_pairs:
        scored.append((similarity(question, faq_question), faq_question, faq_answer))
    scored.sort(reverse=True, key=lambda item: item[0])
    return scored[0] if scored else (0.0, '', '')


faq_pairs = load_faq()
print(f'Jumlah FAQ: {len(faq_pairs)}')
for q, a in faq_pairs:
    print(f'- {q} -> {a}')


### 5) Jalankan Chatbot CLI/Terminal

Chatbot berikut memakai model Ollama Cloud yang Anda pilih. Respons dijaga tetap terbatas pada konteks FAQ yang ditemukan.

Ketik `exit` atau `quit` untuk mengakhiri percakapan.


In [ ]:
client = Client(
    host='https://ollama.com',
    headers=OLLAMA_HEADERS,
)

FALLBACK_ANSWER = 'Maaf, saya tidak dapat membantu dengan pertanyaan itu.'

SYSTEM_PROMPT = (
    'Anda adalah chatbot FAQ yang sangat ketat. ' 
    'Gunakan hanya konteks FAQ yang diberikan. ' 
    'Jangan menambah fakta baru, jangan mengarang, dan jangan menjawab di luar konteks. ' 
    f'Jika konteks tidak cukup, jawab persis: {FALLBACK_ANSWER}'
)


def answer_question(question):
    score, matched_question, matched_answer = find_best_faq_match(question, faq_pairs)
    if score < 0.20:
        return FALLBACK_ANSWER

    context = (
        'Konteks FAQ yang relevan:\n'
        f'Pertanyaan FAQ: {matched_question}\n'
        f'Jawaban FAQ: {matched_answer}\n\n'
        f'Pertanyaan pengguna: {question}\n'
        'Jawab singkat dan hanya berdasarkan jawaban FAQ di atas.'
    )

    try:
        stream = client.chat(
            model=selected_model,
            messages=[
                {'role': 'system', 'content': SYSTEM_PROMPT},
                {'role': 'user', 'content': context},
            ],
            stream=True,
        )
        chunks = []
        for part in stream:
            delta = part.get('message', {}).get('content', '')
            if delta:
                chunks.append(delta)
        response_text = ''.join(chunks).strip()
        return response_text or matched_answer
    except Exception as exc:
        print(f'\n[Fallback karena error Ollama Cloud: {exc}]')
        return matched_answer


def run_chatbot():
    print('\nChatbot FAQ siap digunakan.')
    print(f'Model aktif: {selected_model}')
    print('Ketik exit atau quit untuk keluar.')

    while True:
        user_input = input('\nUser: ').strip()
        if user_input.lower() in {'exit', 'quit'}:
            print('Bot: Terima kasih. Sesi chatbot selesai.')
            break
        if not user_input:
            print('Bot: Silakan masukkan pertanyaan.')
            continue

        print('Bot: ', end='', flush=True)
        response = answer_question(user_input)
        print(response)


run_chatbot()


## Bagian 2 - Teori AI

### Soal 2: Pertanyaan Teori Dasar

**a. Apa yang dimaksud dengan Artificial Intelligence (AI)? Sebutkan dua contohnya dalam kehidupan sehari-hari.**

Artificial Intelligence (AI) adalah bidang ilmu komputer yang membuat mesin mampu meniru kemampuan cerdas manusia, seperti mengenali pola, memahami bahasa, mengambil keputusan, dan belajar dari data.

Dua contoh AI dalam kehidupan sehari-hari:

- Rekomendasi video di YouTube atau Netflix.
- Asisten virtual seperti Siri, Google Assistant, atau ChatGPT.

**b. Apa perbedaan antara Supervised Learning dan Unsupervised Learning? Berikan satu contoh untuk masing-masing.**

- **Supervised Learning** menggunakan data yang sudah memiliki label jawaban. Model belajar dari pasangan input-output yang benar. Contoh: klasifikasi email spam dan bukan spam.
- **Unsupervised Learning** menggunakan data tanpa label. Model mencari pola atau struktur sendiri. Contoh: clustering pelanggan berdasarkan perilaku belanja.

### Soal 3: Pertanyaan Konsep

**a. Apa itu Feature dalam konteks machine learning? Mengapa penting untuk memilih fitur yang tepat saat membangun model?**

Feature adalah variabel atau atribut yang digunakan model sebagai masukan untuk mempelajari pola. Contohnya umur, pendapatan, atau jumlah klik.

Pemilihan fitur yang tepat penting karena fitur yang relevan membantu model belajar lebih akurat, lebih cepat, dan lebih stabil. Fitur yang kurang tepat dapat membuat model sulit belajar atau menghasilkan prediksi yang kurang baik.

**b. Apa itu Fine-tuning dalam machine learning? Sebutkan satu kasus di mana fine-tuning berguna.**

Fine-tuning adalah proses menyesuaikan model yang sudah pre-trained agar lebih cocok dengan tugas atau data yang lebih spesifik.

Contoh kasus yang berguna: model bahasa umum di-fine-tune untuk klasifikasi sentimen ulasan pelanggan pada domain e-commerce atau layanan keuangan.

### Kesimpulan

Notebook ini menunjukkan implementasi chatbot FAQ berbasis Ollama Cloud yang hanya menjawab dari konteks `faq.txt`, serta jawaban teori AI dasar dalam format yang siap dijalankan di Google Colab.
